# Data Burn - Módulo de Applied Computer Vision (ACV)
**FIAP Global Solution 2026**  
*Tema: Indústria Espacial - O Código que Move o Universo*  

Este notebook apresenta o pipeline completo de Inteligência Artificial para o processamento de imagens de satélite/aéreas da nossa solução de monitoramento ambiental. 

### Objetivo da Entrega:
Construir um pipeline de classificação espectral capaz de separar imagens aéreas em 3 categorias críticas:
1. **`smoke` (Fumaça)**: Detecção de colunas de incêndios ativos.
2. **`burned_land` (Área Queimada)**: Cicatrizes e devastação pós-fogo.
3. **`at_risk_vegetation` (Vegetação em Risco)**: Cobertura florestal sob monitoramento preventivo.

**Regra Acadêmica:** É estritamente proibido o uso de *Transfer Learning* (modelos pré-treinados). Ambas as redes convolucionais apresentadas aqui foram desenvolvidas **totalmente do zero**.

--- 
## Passo 1: Configuração do Ambiente e Importações

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from PIL import Image

# Adiciona a pasta src ao PATH para permitir importações dos nossos scripts estruturados
sys.path.append(os.path.abspath('src'))

from dataset import get_dataloaders
from models import FireNet_Lite, SpaceFire_DeepCNN

# Configuração de Hardware acelerado (GPU CUDA ou CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo ativo para treinamento: {device.type.upper()}")

--- 
## Passo 2: Geração do Dataset e Data Augmentation

Abaixo inicializamos os loaders aplicando transformações específicas para imagens espaciais (invariantes a giros gravitacionais):

In [ ]:
# Caso as imagens sintéticas ainda não tenham sido geradas, descomente as linhas abaixo:
# from data.generate_dummy_data import generate_dataset
# generate_dataset('data')

# Carregando os loaders de treino, validação e teste
train_loader, val_loader, test_loader, classes = get_dataloaders(
    data_dir='data', batch_size=32, img_size=(128, 128)
)

--- 
## Passo 3: Arquiteturas CNN Desenvolvidas Do Zero

Revisão rápida das duas redes implementadas no módulo `src/models.py`:
1. **`FireNet_Lite`**: Rede sequencial simples, de rápida convergência e baixo custo computacional. Conta com 3 blocos Conv-BN-ReLU-MaxPool e Dropout de 40%.
2. **`SpaceFire_DeepCNN`**: Arquitetura premium com **Blocos Residuais próprios** implementados do zero. Permite conexões residuais diretas de atalho (*shortcuts*) prevenindo o desvanecimento de gradientes e acelerando a convergência.

In [ ]:
# Instanciação e visualização rápida do número de parâmetros das redes
model_lite = FireNet_Lite(num_classes=len(classes)).to(device)
model_deep = SpaceFire_DeepCNN(num_classes=len(classes)).to(device)

params_lite = sum(p.numel() for p in model_lite.parameters() if p.requires_grad)
params_deep = sum(p.numel() for p in model_deep.parameters() if p.requires_grad)

print(f"-> FireNet_Lite | Parâmetros Treináveis: {params_lite:,}")
print(f"-> SpaceFire_DeepCNN | Parâmetros Treináveis: {params_deep:,}")

--- 
## Passo 4: Loop de Treinamento Interativo

Abaixo definimos o otimizador Adam, a função de perda (CrossEntropyLoss) e executamos o treino da nossa rede campeã residual por 10 épocas, registrando as métricas a cada passo:

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_deep.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 10

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc = 0.0

print("Iniciando o Treinamento da SpaceFire_DeepCNN...")
for epoch in range(1, epochs + 1):
    # Fase de Treinamento
    model_deep.train()
    running_loss, running_corrects, total_samples = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_deep(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        running_corrects += torch.sum(preds == labels.data).item()
        total_samples += inputs.size(0)
        
    epoch_train_loss = running_loss / total_samples
    epoch_train_acc = running_corrects / total_samples
    
    # Fase de Validação
    model_deep.eval()
    running_val_loss, running_val_corrects, total_val_samples = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_deep(inputs)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            running_val_corrects += torch.sum(preds == labels.data).item()
            total_val_samples += inputs.size(0)
            
    epoch_val_loss = running_val_loss / total_val_samples
    epoch_val_acc = running_val_corrects / total_val_samples
    
    # Grava histórico
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"Epoch {epoch:02d}/{epochs:02d} | "
          f"Train Loss: {epoch_train_loss:.4f} - Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} - Acc: {epoch_val_acc*100:.2f}%")
    
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        torch.save(model_deep.state_dict(), 'checkpoints/best_model_deep_notebook.pth')
        print(" ==> Novo checkpoint de pesos salvo!")

--- 
## Passo 5: Avaliação Científica das Métricas

### A. Plotagem das Curvas de Perda e Acurácia por Época

In [ ]:
epochs_range = range(1, epochs + 1)
plt.figure(figsize=(14, 5))

# Gráfico de Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], 'r-o', label='Loss Treino')
plt.plot(epochs_range, history['val_loss'], 'b--s', label='Loss Validação')
plt.title('Evolução da Função de Perda (Loss)')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Gráfico de Acurácia
plt.subplot(1, 2, 2)
plt.plot(epochs_range, [x*100 for x in history['train_acc']], 'g-o', label='Acc Treino')
plt.plot(epochs_range, [x*100 for x in history['val_acc']], 'y--s', label='Acc Validação')
plt.title('Evolução da Acurácia (%)')
plt.xlabel('Época')
plt.ylabel('Acurácia (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### B. Inferência no Conjunto de Teste & Matriz de Confusão

Abaixo carregamos o melhor modelo, fazemos a predição cega nas 90 imagens do conjunto de testes e geramos a Matriz de Confusão interativa com mapa de calor:

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model_deep.load_state_dict(torch.load('checkpoints/best_model_deep_notebook.pth', map_location=device))
model_deep.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model_deep(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = np.mean(all_preds == all_labels) * 100
print(f"Acurácia de Referência no Conjunto de Teste: {accuracy:.2f}% (Meta Mínima: 88%)")

print("\nRelatório de Métricas:")
print(classification_report(all_labels, all_preds, target_names=classes))

# Matriz de Confusão
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes)
plt.title('Matriz de Confusão Científica - SpaceFire_DeepCNN')
plt.xlabel('Classe Predita')
plt.ylabel('Classe Real')
plt.show()